# Colonias UCU – UdelaR: entrenar y evaluar en Colab

Este notebook **no tiene lógica propia**: clona el repo y llama a los mismos scripts que se usan en una computadora local, así los resultados salen del mismo código.

**Antes de empezar:**
1. `Entorno de ejecución → Cambiar tipo de entorno → GPU T4`.
2. Cargar la API key de Roboflow en los *Secrets* de Colab (ícono de llave 🔑, nombre `ROBOFLOW_API_KEY`). **Nunca escribirla en una celda.**
3. Elegir el experimento en la celda 3 y ejecutar todo en orden.

Si Colab se desconecta: volver a ejecutar todas las celdas. Las corridas terminadas se saltean y la que quedó a medias se retoma desde la última época copiada a Drive.

## 1 · GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 · Repo y dependencias

In [ ]:
REPO = 'https://github.com/<org>/colonias-ucu-udelar.git'   # <- completar
!git clone -q $REPO /content/colonias 2>/dev/null || git -C /content/colonias pull -q
%cd /content/colonias
!pip install -q -e .
import ultralytics; print('ultralytics', ultralytics.__version__)   # esperado: 8.4.163

## 3 · Experimento

| Config | Qué reproduce | Tiempo aprox. (T4) |
|---|---|---|
| `v8` | Modelo de referencia `best_v8.pt` | ~13 min |
| `mlca_v5_base`, `mlca_v5_mlca` | YOLOv8m vs. YOLOv8m + MLCA, 3 seeds cada uno | ~40 min cada uno |
| `agar_base`, `agar_mlca` | YOLOv8m vs. YOLOv8m + MLCA sobre AGAR | ~4 h y ~5 h |
| `v1` … `v7` | Versiones anteriores (ver `docs/versiones.md`) | 6 min – 2 h |

In [ ]:
EXPERIMENTO = 'v8'
SEEDS = ''            # vacío = las del config; p. ej. '0 1 2'

## 4 · Google Drive (para no perder las corridas)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SYNC = '/content/drive/MyDrive/colonias/runs'

## 5 · Entrenar
Entrena en el disco local de Colab y copia a Drive cada 2 épocas (entrenar directo en Drive falla con `OSError: [Errno 95]`).

In [ ]:
seeds = f'--seeds {SEEDS}' if SEEDS else ''
!python -m colonias.train configs/experiments/{EXPERIMENTO}.yaml {seeds} --project /content/runs --sync {SYNC}

## 6 · Evaluar y comparar con los resultados esperados

In [ ]:
!python -m colonias.evaluate configs/experiments/{EXPERIMENTO}.yaml --project /content/runs
!python scripts/tabla_resultados.py

## (Opcional) Evaluar los pesos publicados, sin reentrenar

Bajar `best_v8.pt` del Release del repo y evaluarlo: reproduce las métricas al decimal.

In [ ]:
# !wget -q https://github.com/<org>/colonias-ucu-udelar/releases/download/v1.0-informe/best_v8.pt
# !python -m colonias.evaluate configs/experiments/v8.yaml --pesos best_v8.pt